# Notebook 06 — AnomalyDINO MVTec AD Experiments

## Overview

This notebook contains all AnomalyDINO experiments on MVTec AD.
It uses Anomalib 2.3.3 which is required for correct single-class
performance. Anomalib 2.4.0 produces near-random scores for single-class
evaluation due to a change in the internal scoring pipeline.

## Experiments

### 1. Single-Class Validation (Section 3.2.2, Investigation 1)
AnomalyDINO evaluated per-category with a single memory bank per category.
16-shot protocol, 3 runs averaged. Establishes the single-class baseline
against which multi-class degradation is measured.

### 2. Multi-Class Validation (Investigation 1)
AnomalyDINO evaluated with all 15 categories sharing one memory bank.
16-shot per category (240 total reference images). Tests whether the
near-random performance on Real-IAD reflects a general multi-class
limitation or a dataset-specific characteristic.

### 3. Investigation 4 — Category Scaling on MVTec AD
AnomalyDINO evaluated at 5, 10, and 15 categories (shared memory bank).
The 15-category result comes from the multi-class validation above.
Results saved to: results/ablation/investigation4/mvtec/

## Note on Real-IAD AnomalyDINO Experiments
The factorial design ablation (Investigation 1) on Real-IAD is conducted
in notebook 04 using the train_anomalydino_fewshot trainer function.
All results are unified in the analysis notebook 05 via shared Drive paths.

In [ ]:
import os
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'
from google.colab import drive
import sys
drive.mount('/content/drive')

repo_path = '/content/drive/MyDrive/BachelorsThesis'
results_path = f'{repo_path}/results'
sys.path.insert(0, repo_path)

!pip install anomalib==2.3.3 ADEval einops timm kornia -q

import torch
import numpy as np
import pandas as pd
import gc
from pathlib import Path
from sklearn.metrics import roc_auc_score

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
if not os.path.exists('/content/mvtec/bottle'):
    from anomalib.data import MVTecAD
    dm = MVTecAD(root='/content/mvtec', category='bottle')
    dm.prepare_data()
    print('MVTec AD downloaded')
else:
    print('MVTec AD already present')

In [ ]:
from anomalib.models import AnomalyDINO
from anomalib.data import MVTecAD
from anomalib.data.utils.split import TestSplitMode
from anomalib.engine import Engine
from anomalib.pre_processing import PreProcessor
from torch.utils.data import DataLoader, Subset
import torchvision.transforms.v2 as T2

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
N_SHOTS = 16
N_RUNS = 3

MVTEC_CATS = [
    'bottle', 'cable', 'capsule', 'carpet', 'grid',
    'hazelnut', 'leather', 'metal_nut', 'pill', 'screw',
    'tile', 'toothbrush', 'transistor', 'wood', 'zipper'
]
MVTEC_5  = MVTEC_CATS[:5]
MVTEC_10 = MVTEC_CATS[:10]

transform_448 = T2.Compose([
    T2.Resize((448, 448),
              interpolation=T2.InterpolationMode.BICUBIC,
              antialias=True),
    T2.Normalize(mean=[0.485, 0.456, 0.406],
                 std=[0.229, 0.224, 0.225]),
])

def collate_image_items(batch):
    return torch.stack([item.image for item in batch])

# Map save paths
maps_sc  = f'{results_path}/anomaly_maps/mvtec/anomalydino_singleclass'
maps_mc  = f'{results_path}/anomaly_maps/mvtec/anomalydino_multiclass'
maps_5c  = f'{results_path}/anomaly_maps/mvtec/anomalydino_5cat'
maps_10c = f'{results_path}/anomaly_maps/mvtec/anomalydino_10cat'
abl4_mvtec = f'{results_path}/ablation/investigation4/mvtec'

for p in [maps_sc, maps_mc, maps_5c, maps_10c, abl4_mvtec]:
    os.makedirs(p, exist_ok=True)

print('Setup complete')

In [ ]:
def build_memory_bank(categories, run_idx=0):
    """Build AnomalyDINO memory bank for given categories.
    Single-class: pass one category. Multi-class: pass multiple.
    run_idx controls which 16-shot window to use (0=1-16, 1=17-32, 2=33-48).
    """
    model = AnomalyDINO(
        encoder_name='dinov2reg_vit_base_14',
        coreset_subsampling=False,
        masking=False,
    )
    model.pre_processor = PreProcessor(transform=transform_448)
    torch_model = model.model.to(DEVICE)
    torch_model.train()

    for cat in categories:
        datamodule = MVTecAD(
            root='/content/mvtec',
            category=cat,
            train_batch_size=32,
            eval_batch_size=8,
            num_workers=2,
            test_split_mode=TestSplitMode.FROM_DIR,
        )
        datamodule.prepare_data()
        datamodule.setup()
        train_dataset = datamodule.train_data
        start = run_idx * N_SHOTS
        indices = list(range(start, min(start + N_SHOTS, len(train_dataset))))
        shot_dataset = Subset(train_dataset, indices)
        shot_loader = DataLoader(
            shot_dataset, batch_size=16, shuffle=False,
            num_workers=0, collate_fn=collate_image_items
        )
        with torch.no_grad():
            for images in shot_loader:
                images = model.pre_processor.transform(images).to(DEVICE)
                torch_model(images)

    torch_model.fit()
    model.model = torch_model
    return model


def run_inference_and_save_maps(model, categories, maps_dir):
    """Run inference, save anomaly maps, return per-category I-AUROC."""
    rows = []
    model.model.eval()
    for cat in categories:
        datamodule = MVTecAD(
            root='/content/mvtec',
            category=cat,
            eval_batch_size=8,
            num_workers=2,
            test_split_mode=TestSplitMode.FROM_DIR,
        )
        datamodule.prepare_data()
        datamodule.setup()

        cat_map_dir = f'{maps_dir}/{cat}'
        os.makedirs(cat_map_dir, exist_ok=True)

        scores, labels = [], []
        with torch.no_grad():
            for batch in datamodule.test_dataloader():
                images = model.pre_processor.transform(
                    torch.stack([item.image for item in batch])
                ).to(DEVICE)
                gt = torch.tensor(
                    [item.gt_label for item in batch]).numpy()
                out = model.model(images)
                pred_scores = out.pred_score.cpu().numpy().flatten()
                scores.extend(pred_scores)
                labels.extend(gt.flatten())
                for amap in out.anomaly_map.cpu().numpy():
                    idx = len(os.listdir(cat_map_dir))
                    np.save(f'{cat_map_dir}/{cat}_{idx:04d}.npy', amap)

        auroc = roc_auc_score(labels, scores) * 100
        print(f'  {cat}: {auroc:.2f}%')
        rows.append({'Category': cat, 'i_auroc': round(auroc, 2)})
    return pd.DataFrame(rows)

def run_inference_no_maps(model, categories):
    rows = []
    model.model.eval()
    for cat in categories:
        datamodule = MVTecAD(
            root='/content/mvtec',
            category=cat,
            eval_batch_size=8,
            num_workers=2,
            test_split_mode=TestSplitMode.FROM_DIR,
        )
        datamodule.prepare_data()
        datamodule.setup()
        scores, labels = [], []
        with torch.no_grad():
            for batch in datamodule.test_dataloader():
                images = model.pre_processor.transform(
                    torch.stack([item.image for item in batch])
                ).to(DEVICE)
                gt = torch.tensor(
                    [item.gt_label for item in batch]).numpy()
                out = model.model(images)
                scores.extend(out.pred_score.cpu().numpy().flatten())
                labels.extend(gt.flatten())
        auroc = roc_auc_score(labels, scores) * 100
        print(f'  {cat}: {auroc:.2f}%')
        rows.append({'Category': cat, 'i_auroc': round(auroc, 2)})
    return pd.DataFrame(rows)

print('Helper functions ready')

## 1. Single-Class Protocol (Baseline)

One memory bank per category, 16 reference images per category.
This matches AnomalyDINO's primary published evaluation setting.
Three runs with different reference image sets are averaged to reduce
variance from reference sample selection (Run 1: images 1-16,
Run 2: images 17-32, Run 3: images 33-48).

Published reference (672px, ViT-Small): 98.4% I-AUROC on MVTec AD (Hofer et al., 2025).
This experiment uses 448px and ViT-Base for consistency with the thesis backbone.

In [ ]:
print('=== Single-class protocol (3 runs, 16-shot) ===')
sc_run_results = {cat: [] for cat in MVTEC_CATS}

for run_idx in range(N_RUNS):
    save_maps = (run_idx == 0)
    print(f'\nRun {run_idx + 1}/{N_RUNS} — maps: {save_maps}')
    for cat in MVTEC_CATS:
        model = build_memory_bank([cat], run_idx=run_idx)
        maps_dir = maps_sc if save_maps else None
        df_run = run_inference_and_save_maps(model, [cat], maps_dir) \
            if save_maps else run_inference_no_maps(model, [cat])
        sc_run_results[cat].append(df_run['i_auroc'].values[0])
        del model
        torch.cuda.empty_cache()
        gc.collect()

sc_rows = []
for cat in MVTEC_CATS:
    runs = sc_run_results[cat]
    sc_rows.append({
        'Category': cat,
        'Run 1': round(runs[0], 2),
        'Run 2': round(runs[1], 2),
        'Run 3': round(runs[2], 2),
        'Single-class I-AUROC': round(np.mean(runs), 2),
        'SC Std': round(np.std(runs), 2),
    })
sc_df = pd.DataFrame(sc_rows)
sc_df.to_csv(f'{results_path}/mvtec_anomalydino_singleclass.csv', index=False)
print(f'\nSingle-class mean I-AUROC: {sc_df["Single-class I-AUROC"].mean():.2f}%')
print(sc_df[['Category', 'Single-class I-AUROC', 'SC Std']].to_string(index=False))
print('Saved: mvtec_anomalydino_singleclass.csv')

## 2. Multi-Class Protocol

One shared memory bank across all 15 categories, 16 reference images per category
(240 total reference images). This mirrors the Real-IAD standard protocol where
all categories share a single memory bank.

In [ ]:
print('=== Multi-class protocol (15 categories, 16-shot shared bank) ===')
model_mc = build_memory_bank(MVTEC_CATS, run_idx=0)
mc_df = run_inference_and_save_maps(model_mc, MVTEC_CATS, maps_mc)
mc_df.columns = ['Category', 'Multi-class I-AUROC']

comparison = mc_df.merge(
    sc_df[['Category', 'Single-class I-AUROC', 'SC Std']], on='Category')
comparison['Difference (MC - SC)'] = (
    comparison['Multi-class I-AUROC'] - comparison['Single-class I-AUROC']
).round(2)
comparison.to_csv(
    f'{results_path}/mvtec_anomalydino_multiclass_comparison.csv', index=False)
print(f'\nMulti-class mean I-AUROC: {mc_df["Multi-class I-AUROC"].mean():.2f}%')
print(f'Mean degradation: {comparison["Difference (MC - SC)"].mean():.2f}pp')
print('Saved: mvtec_anomalydino_multiclass_comparison.csv')
del model_mc
torch.cuda.empty_cache()
gc.collect()

## 3. Investigation 4 — Category Scaling on MVTec AD

This section evaluates how AnomalyDINO's detection performance changes
as the number of categories sharing a single memory bank increases.
Three configurations are tested: 5, 10, and 15 categories.

Category subsets are defined alphabetically and are strictly nested:
the 5-category subset is contained within the 10-category subset,
which is contained within the full 15-category set. This ensures
that any observed performance change is attributable to the increase
in category count rather than differences in which categories are included.

The 15-category result is taken directly from the multi-class validation
above. The 5 and 10-category configurations are run here with the same
16-shot protocol and shared memory bank structure.

All results are saved to: results/ablation/investigation4/mvtec/
and are loaded by the analysis notebook 05 alongside the Real-IAD
scaling results for Dinomaly and INP-Former.

In [ ]:
print('=== Investigation 4: 5 categories (shared bank) ===')
model_5cat = build_memory_bank(MVTEC_5, run_idx=0)
df_5cat = run_inference_and_save_maps(model_5cat, MVTEC_5, maps_5c)
df_5cat.to_csv(f'{abl4_mvtec}/anomalydino_5cat_scores.csv', index=False)
print(f'Mean I-AUROC 5-cat: {df_5cat["i_auroc"].mean():.2f}%')
del model_5cat
torch.cuda.empty_cache()
gc.collect()

In [ ]:
print('=== Investigation 4: 10 categories (shared bank) ===')
model_10cat = build_memory_bank(MVTEC_10, run_idx=0)
df_10cat = run_inference_and_save_maps(model_10cat, MVTEC_10, maps_10c)
df_10cat.to_csv(f'{abl4_mvtec}/anomalydino_10cat_scores.csv', index=False)
print(f'Mean I-AUROC 10-cat: {df_10cat["i_auroc"].mean():.2f}%')
del model_10cat
torch.cuda.empty_cache()
gc.collect()

## 4. Comparison Summary

In [ ]:
comparison = mc_df.merge(
    sc_df[['Category', 'Single-class I-AUROC', 'SC Std']],
    on='Category'
)
comparison['Difference (MC - SC)'] = (
    comparison['Multi-class I-AUROC'] - comparison['Single-class I-AUROC']
).round(2)

print('=' * 72)
print('AnomalyDINO: Multi-class vs Single-class on MVTec AD (16-shot, 448px)')
print('=' * 72)
print(comparison[['Category', 'Single-class I-AUROC', 'SC Std',
                   'Multi-class I-AUROC', 'Difference (MC - SC)']].to_string(index=False))
print()
print(f'Mean single-class I-AUROC: {comparison["Single-class I-AUROC"].mean():.2f}%')
print(f'Mean multi-class I-AUROC:  {comparison["Multi-class I-AUROC"].mean():.2f}%')
print(f'Mean degradation (MC-SC):  {comparison["Difference (MC - SC)"].mean():.2f}pp')
print()
print('Published reference (672px, ViT-Small): 98.4% I-AUROC (Damm et al., 2025)')

comparison.to_csv(
    f'{results_path}/mvtec_anomalydino_multiclass_comparison.csv',
    index=False)
print('\nSaved: mvtec_anomalydino_multiclass_comparison.csv')